In [1]:
"""ATLAS's Circuit Shaper Simulator"""

import os
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import sympy as sp
from sympy import Eq
from sympy import fraction
from sympy.abc import s
from scipy import signal
from scipy.stats import pearsonr
import random

In [ ]:
## Matplotlib Configuration ##

plt.rcParams.update(
    {
        "font.size": 20,  # Tamanho da fonte
        "figure.figsize": (10, 6),  # Tamanho da figura
        # "font.family": "Arial",  # ou "serif", "sans-serif", "Times New Roman", etc.
        "axes.titlesize": 17,  # Título do gráfico
        "axes.labelsize": 17,  # Títulos dos eixos
        "xtick.labelsize": 13,  # Ticks do eixo X
        "ytick.labelsize": 13,  # Ticks do eixo Y
        "legend.fontsize": 13,  # Legendas
    }
)

In [ ]:
# Constantes #
t1 = np.arange(0, 400) * 25 * 10**-9
xlabels = ["C0", "Ca", "Cb", "Cc", "La", "Lb", "Lc", "RL"]
pause = 1

In [2]:
## Functions auxiliares ##


def ckt_parameters():
    # print("ckt_parameters")

    ## Definition of constants ##
    tau_1, tau_2, Vo, Vi = sp.symbols("tau_1 tau_2 Vo Vi")
    CC0, C1, C2, C3 = sp.symbols("CC0 C1 C2 C3")
    RL = sp.symbols("RL")
    L1, L2, L3 = sp.symbols("L1 L2 L3")
    I1, I2, I3, I4 = sp.symbols("I1 I2 I3 I4")

    # Ckt components in order
    Cord = [CC0, C1, C2, C3, L1, L2, L3, RL, tau_1, tau_2]

    # Nominal values (old values) of Cord elements in order
    Cval = [
        100e-9,  # CC0
        120e-12,  # Ca = C1
        130e-12,  # Cb = C3 + C2
        83e-12,  # Cc = C5 + C4
        2.48e-6,  # La = L1 + L2
        1.6e-6,  # Lb = L3 + L4
        0.78e-6,  # Lc = L5 + L6
        138.8338,  # RL = (R1 + R2) //  R3
        3.1046e-09,  # tau_2
        6.5798e-09,  # tau_1
    ]
    # Ckt equations
    eqn1 = Eq(I1 / (CC0 * s) + (I1 - I2) / (C1 * s), Vi)
    eqn2 = Eq((I2 - I3) / (C2 * s) - (I1 - I2) / (C1 * s) + L1 * s * I2, 0)
    eqn3 = Eq((I3 - I4) / (C3 * s) - (I2 - I3) / (C2 * s) + L2 * s * I3, 0)
    eqn4 = Eq(I4 * RL - (I3 - I4) / (C3 * s) + L3 * s * I4, 0)
    eqn5 = Eq(I4 * RL, Vo)

    eqns = [eqn1, eqn2, eqn3, eqn4, eqn5]

    # Solver
    Sol = sp.solve(eqns, (I1, I2, I3, I4, Vo))
    si6 = Sol[I4]

    # PMT
    PMT = (1 / tau_1 - 1 / tau_2) / (
        s**2 + (1 / tau_1 + 1 / tau_2) * s + 1 / tau_1 / tau_2
    )

    # I_out without V_in
    h = RL * si6 / Vi

    # Final transfer function
    H1 = PMT * h

    return H1, Cord, Cval


def MonteCarlo_iteration(
    iterations,
    erro,
    components,
    nominal_values,
    FT,
    t,
    seed: int | None = None,
):
    """
    iter = 0 represents pure/error-free signal value
    MonteCarlo[0] = actual/error-free signal value

    FT: Transfer Function
    t1: time vector
    iterations: int
        number of iterations
    erro: percentage error of circuit components
    nominal_values: nominal values of circuit components

     seed: int | None
        Semente opcional para reprodutibilidade.
    """
    # print("Monte Carlo iteration")

    # Store all the poles of the iterations
    all_poles = []

    # Store the values ​​with error of each iteration
    MonteCarlo = []

    # Store all the FPs added of each iteration # list of all the graphs added with errors
    y_out = []

    # Helper to store the values ​​of y
    y1 = []

    # Seed para reprodutibilidade
    rng = random.Random(seed)

    for iter in range(iterations):

        # Random values; # error range; maximum error from -e% to +e%
        xa = []  # FP function of the summed iteration
        new_Cval = []  # List of components with changed values

        # Changing element values ​​without changing tau1 and tau2
        new_Cval = [
            (
                value * (rng.gauss(0, erro[idx])) + value
                if iter != 0 and idx < len(nominal_values) - 2
                else value
            )
            for idx, value in enumerate(nominal_values)
        ]

        # Saving component variations
        MonteCarlo.append(new_Cval[:-2])

        H = FT
        for variavel, v in zip(components, new_Cval):
            H = H.subs(variavel, v)

        # Separating numerator from denominator
        N_H, D_H = fraction(H)

        """RESIDUOS E POLOS"""

        coefs_num = []  # reset variable
        coefs_den = []  # reset variable

        coefs_num = sp.Poly(N_H, s).all_coeffs()  # get coefs
        coefs_den = sp.Poly(D_H, s).all_coeffs()  # get coefs

        # Frações parciais
        residuos, polos, b0 = [], [], []
        residuos, polos, b0 = signal.residue(coefs_num, coefs_den)

        # Saving all poles
        all_poles.append(polos)

        # Residual correction (removing the img part 0j from the real residuals)
        for polo1, residuo1 in zip(polos, residuos):
            if polo1.imag == 0:
                residuo1 = residuo1.real

        """LAPLACE INVERSA E GRAFICOS"""

        for enum, (polo, residuo) in enumerate(zip(polos, residuos)):

            # Check if the imaginary part is zero
            if polo.imag == 0:
                "EXPONENCIAIS"

                # Removing the img part (0j) of the real residues
                A = residuo.real
                d = polo.real
                x = A * np.exp(d * t)

                xa.append(x)

            else:
                "SENOS E COSSENOS"

                pol_1 = polos[enum - 1]  # conjugate

                if polo != pol_1 and polo != np.conjugate(pol_1):

                    a1 = polo.real
                    b1 = abs(polo.imag)

                    Mod = abs(residuos[enum])  # Modulo
                    fase = np.angle(residuos[enum])  # fase in rad

                    # FP term
                    x = 2 * Mod * np.exp(a1 * t) * np.cos(b1 * t + fase)

                    xa.append(x)

        "SUMMING THE TERMS"

        if iter != 0:
            y1 = sum(xa).real / maxs
            y_out.append(y1)

        else:
            sinal0 = sum(xa).real
            maxs = np.max(np.abs(sinal0))
            y1 = sinal0 / maxs  # largest module/ normalize
            y_out.append(y1)

    return np.array(MonteCarlo).T, all_poles, y_out


def config_plot_pulso(titles, labelx, ylabel, xlim, ylim=None):
    """
    Configures the visual appearance of a Matplotlib plot for pulse signals.

    Adds title, axis labels, limits, axis lines at zero, and a grid to the plot.

    Args:
        titulo (str): The title of the plot.
        labelx (str): The label for the x-axis.
        ylabel (str): The label for the y-axis.
        xlim (tuple): Tuple containing the minimum and maximum values for the x-axis (x_min, x_max).
        ylim (tuple, optional): Tuple containing the minimum and maximum values for the y-axis (y_min, y_max). Defaults to None.

    Returns:
        None
    """
    plt.title(titles)
    plt.xlabel(labelx)
    plt.ylabel(ylabel)
    plt.xlim(*xlim)
    if ylim:
        plt.ylim(*ylim)
    plt.axhline(0, color="black", linewidth=0.8)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.grid(True)
    plt.tight_layout()


def plot_pulso(
    t,
    y,
    sigma,
    show_fig=False,
    only_bandas=True,
    save_directory="Pulse",
    file_name="Banda_incerteza",
    save_fig=False,
):
    print("Plot pulse")

    # Mean and standard deviation calculation
    y = np.array(y)
    ymed = np.mean(y, axis=0)
    desv_pad = np.std(y, axis=0)

    if not only_bandas:
        # --- 1. Pulse with variations ---
        [plt.plot(t, yi, color="b", linewidth=2) for yi in y]
        config_plot_pulso(
            "Pulso com Variação", "Tempo", "Intensidade do Pulso", (-0.25e-6, 0.5e-6)
        )

        if show_fig:
            plt.show(block=False)
            plt.pause(pause - pause / 2)
            plt.close()
            plt.clf()

        # --- 2. Pulso without variations ---
        plt.plot(t, y[0], color="b", linewidth=2)
        config_plot_pulso(
            "Pulso Sem Variação", "Tempo", "Intensidade do Pulso", (-0.25e-6, 0.5e-6)
        )

        if show_fig:
            plt.show(block=False)
            plt.pause(pause - pause / 2)
            plt.close()
            plt.clf()

    # --- 3. Uncertainty Band ---
    banda_sup = ymed + desv_pad * sigma
    banda_inf = ymed - desv_pad * sigma

    plt.plot(y[0], label="Pulso com Valores Nominais", color="b", linewidth=2)
    plt.plot(ymed, label="Média dos Erros", linestyle="--", color="orange", linewidth=2)
    plt.plot(banda_sup, label="Banda Superior", linestyle="-.", color="g", linewidth=2)
    plt.plot(
        banda_inf,
        label="Banda Inferior",
        linestyle="-.",
        color="r",
        linewidth=2,
    )
    plt.fill_between(
        range(len(banda_sup)), banda_sup, banda_inf, color="gray", alpha=0.5
    )

    config_plot_pulso(
        "Bandas de Incerteza", "Tempo", "Intensidade Pulso", (-0.1, 10), (-0.2, 1.25)
    )
    plt.legend()

    if show_fig:
        plt.show(block=False)
        plt.pause(pause)
        plt.close("all")
        plt.clf()